# Workshop 4: Basic Skew-Ts and Station Plots with MetPy

MetPy is a general-purpose toolkit for reading, visualizing, and performing calculations with weather data (and atmospheric data more generally). This notebook will briefly showcase two of the helpful meteorological plot types that MetPy contributes to the Python ecosystem: Skew-T Log-p diagrams (for sounding data) and Station Plots (for maps of observed data at the surface or a given vertical level).

For more information on what else MetPy can help you with, check out [the online documentation](https://unidata.github.io/MetPy/latest/userguide/startingguide.html).

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import metpy.calc as mpcalc
import metpy.plots as mpplots
from metpy.units import units, pandas_dataframe_to_unit_arrays
from metpy.io import add_station_lat_lon

pd.set_option('future.no_silent_downcasting', True)

## Skew-Ts

*For more detailed information on Skew-Ts, I'd recommend [COMET MetEd's module on the topic](https://www.meted.ucar.edu/education_training/lessons/225) as well as [this chapter in Kevin Goebbert's WxTech online textbook](https://kgoebber.github.io/metpy_thebook/weather_technology/skewt/skewt_intro.html).*

Skew-T Log-p diagrams (often simply called *Skew-Ts*) are one of the most commonly used thermodynamic diagrams (especially in the U.S.) in assessing vertical profiles of atmospheric data, often with respect to stability. So, first we're going to need some sounding data to work with! We will pull the data directly from the [Iowa Environmental Mesonet (IEM)](https://mesonet.agron.iastate.edu), and read it using pandas.

In [ ]:
station = "KOAX"
time = pd.Timestamp("2024-07-24 00:00 Z")
df_outer = pd.read_json(
    f"https://mesonet.agron.iastate.edu/json/raob.py?station={station}&ts={time.strftime('%Y-%m-%dT%H:%M:%SZ')}",
)
# Small trick to get to profile within the full output
df = pd.DataFrame.from_records(df_outer.iloc[0,0]['profile'])
df

Now, we need to make sure to prepare our data. In this case, this means cleaning our data, in case there are any NaN values in these data, as well as converting to unit arrays (NumPy arrays with units attached using the Pint library), as MetPy's routines expect.

In [ ]:
# Drop any rows with all NaN values for T, Td, winds
df = df.dropna(
    subset=('pres', 'tmpc', 'dwpc', 'drct', 'sknt'),
    how='all'
).reset_index(drop=True)
data = pandas_dataframe_to_unit_arrays(
    df,
    {'pres': 'hPa', 'hght': 'm', 'tmpc': 'degC', 'dwpc': 'degC', 'drct': 'degrees', 'sknt': 'knots'}
)
# Preview just one of our data arrays
data['tmpc']

One final step before we get to our plot: we need the wind vector expressed as $(u,v)$ components, rather than speed and direction. MetPy has an easy calculation for this, `mpcalc.wind_components`:

In [ ]:
data['u'], data['v'] = mpcalc.wind_components(data['sknt'], data['drct'])
# and, previewing
data['u']

Now, let's actually use MetPy's Skew-T interface to make a basic plot. For more details on all your options, see [the MetPy documentation](https://unidata.github.io/MetPy/latest/tutorials/upperair_soundings.html).

In [ ]:
# Create a new figure. The dimensions here give a good aspect ratio
fig = plt.figure(figsize=(9, 9))
skew = mpplots.SkewT(fig)

# Plot the data using normal plotting functions, in this case using
# log scaling in Y, as dictated by the typical meteorological plot
skew.plot(data['pres'], data['tmpc'], 'tab:red', linewidth=2)
skew.plot(data['pres'], data['dwpc'], 'tab:blue', linewidth=2)
skew.plot_barbs(data['pres'][::4], data['u'][::4], data['v'][::4])

# Show the plot
plt.show()

## Station Plots

As seen above, the basic atmospheric quantities we expected to find in a sounding (for upper air observations) at the very least includes temperature, humidity, and wind (with pressure and/or height as a vertical coordinate). These basic variables are also what we'd expect to find with surface observations. When we have many such observations at different points but at the same level/height (rather than at many levels/heights above a single given point), we often want to plot them on a map, using something called a station model, which looks a bit like the following:

<div style="text-align:center;"><img src="https://kgoebber.github.io/metpy_thebook/_images/surface_station_model.png" style="background:white;max-width:400px;"></div>

The key features to note are the wind barb (indicating wind speed and direction), the temperature indicated in the upper left and the dew point indicated in the lower left. For more details on what the other components mean, and more detailed aspects on how station models are used beyond basic plots like we showcase below, see [these chapters in the aforementioned WxTech text](https://kgoebber.github.io/metpy_thebook/weather_technology/surface_plots/surface_obs_intro.html).

First, as before, we need some data! This will request all ASOS network data from the last hour (so we'll definitely need to do some filtering before using it).

In [ ]:
df = pd.read_csv("https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py?hours=1&latlon=yes", parse_dates=['valid'])
df.head()

Using pandas, let's first select the stations within (and close to) Colorado, then pick the observation closest to the nearest whole hour, and finally filter out any NaNs in important fields. The first part should be fairly self-explanitory, as it is simply setting a bounding box of lat and lon values. The third is as well (as seen with the sounding data in the Skew-T section above). However, the second part is a little bit more complicated, so let's break it down.

We want only one observation for each station, and we want the one that is closest to the nearest whole hour. So, we first get the nearest whole hour (`target_time`). Then, we compute the difference between the valid time and target time for all entries (`df_co_all_times['valid'] - target_time`). However, this gives us a Timedelta object, but we just want an absolute value of seconds, so we need the `.dt.total_seconds().abs()` to do so. Then, we use a groupby operation (in groups based on the `'station'` ID) to perform an operation on each cluster of entries for each station, then within that, pick the index of the smallest time difference (`idxmin`). After all that, we now have a pandas Series of the index of the entry with closest to the target time for each station. So, we can simply index on that `df.loc[...].reset_index(drop=True)`.

In [ ]:
# subset locations
df_co_all_times = df[(df['lat'] >= 36.5) & (df['lat'] <= 41.5) & (df['lon'] <= -101.4) & (df['lon'] >= -109.8)]
# subset to nearest time
target_time = df_co_all_times['valid'].max().floor('h')
df_single_time = df.loc[
    (df_co_all_times['valid'] - target_time).dt.total_seconds().abs().groupby(df_co_all_times['station']).idxmin()
].reset_index(drop=True)
# drop nans (and 'M' flags)
df_filtered = df_single_time.replace('M', np.nan).dropna(how='any', subset=['tmpf', 'dwpf', 'drct', 'sknt'])
df_filtered.head()

Now, we need to do a little bit of translation to get the data from what IEM provides to what MetPy expects for its station plot...particularly because pandas is currently storing the values as strings/objects rather than floats like we need, and some of the units are not what we need either.

In [ ]:
data = {}
data['longitude'] = df_filtered['lon'].values.astype(np.float64)
data['latitude'] = df_filtered['lat'].values.astype(np.float64)
data['air_temperature'] = (df_filtered['tmpf'].values.astype(np.float64) * units.degF).to('degC')
data['dew_point_temperature'] = (df_filtered['dwpf'].values.astype(np.float64) * units.degF).to('degC')
data['eastward_wind'], data['northward_wind'] = mpcalc.wind_components(
    df_filtered['sknt'].values.astype(np.float64) * units.knots,
    df_filtered['drct'].values.astype(np.float64) * units.degrees
)

One final step before making the plot: many of the stations are clustered too close together to be legible. So, let's use a filtering function in MetPy to clear out some of the overlaping stations.

In [ ]:
# Create, then use the Cartopy map projection to transform station locations to the map and
# then refine the number of stations plotted by setting a 60km radius
proj = ccrs.LambertConformal(
    central_longitude=-105, central_latitude=40, standard_parallels=[40]
)
point_locs = proj.transform_points(
    ccrs.PlateCarree(), data['longitude'],  data['latitude']
)
mask = mpcalc.reduce_point_density(point_locs, 6.0e4)
data = {col_name: col_values[mask] for col_name, col_values in data.items()}

Now, let's finally make the plot!

In [ ]:
# Create the figure and an axes set to the projection
fig = plt.figure(figsize=(16, 8))
ax = fig.add_subplot(1, 1, 1, projection=proj)

# Add some various map elements to the plot to make it recognizable
ax.add_feature(cfeature.LAND)
ax.add_feature(cfeature.OCEAN)
ax.add_feature(cfeature.LAKES)
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.STATES)
ax.add_feature(cfeature.BORDERS, linewidth=2)

# Set plot bounds
ax.set_extent((-110.3, -100.9, 36.0, 42.0))

#
# Here's the actual station plot
#

# Start the station plot by specifying the axes to draw on, as well as the
# lon/lat of the stations (with transform). We also the fontsize to 12 pt.
stationplot = mpplots.StationPlot(ax, data['longitude'], data['latitude'],
                          transform=ccrs.PlateCarree(), fontsize=12)

# The layout knows where everything should go, and things are standardized using
# the names of variables. So the layout pulls arrays out of `data` and plots them
# using `stationplot`.
mpplots.simple_layout.plot(stationplot, data)

plt.show()